# Fathom 3 Pre-processing

This notebook provides comprehensive pre-processing tools for Fathom 3 flood hazard datasets.

## About Fathom 3 Data

Fathom 3 provides global flood hazard maps with the following data encoding:

- **Values 0–1,000**: Represent flood depth in **centimetres**. Divide by 100 to convert to metres.
- **Value −32,767**: Indicates **permanent water bodies** (rivers, lakes, ocean).
- **Value −32,768**: Is the **NoData sentinel** — cells without valid flood estimates.
- **Zero values**: Indicate **dry land** for the given return period — valid data, not NoData.

### NoData Management

**CRITICAL**: For all processing operations in this notebook:
- Any **negative value** should be **excluded from computations** (MAX, resampling, statistics)
- Only values **≥ 0** represent valid flood depth data
- Zero is valid data (dry land), not NoData
- Negative values (−32,767 and −32,768) must be preserved in outputs but excluded from calculations

## Three Pre-processing Workflows

This notebook combines three essential Fathom 3 pre-processing workflows:

1. **Tiles Mosaicing**: Merge downloaded Fathom tiles into unified rasters per return period
2. **Flood Type Combination**: Combine different flood types (fluvial, pluvial, coastal) using MAX criteria
3. **Resampling**: Resample rasters from 30m (≈0.000278°) to 90m (≈0.000833°) resolution

---

## Import Libraries

In [ ]:
import os
import time
import shutil
import concurrent.futures
import multiprocessing
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Raster processing libraries
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling

# GUI libraries (optional)
try:
    import ipywidgets as widgets
    import tkinter as tk
    from tkinter import filedialog
    GUI_AVAILABLE = True
except ImportError:
    GUI_AVAILABLE = False
    print("GUI libraries not available. Use manual mode instead.")

# Import custom modules
import merge_utils
import tiff_resampler

print("Libraries imported successfully")

---

# Workflow 1: Tiles Mosaicing

Merge downloaded Fathom tiles into unified rasters per return period.

## Instructions

- Specify the directory (`year` or `SSP scenario`) that contains the RP subfolders — **not** the RP subfolder itself (1in`YEARS`)!
- The output is created in the parent directory of each RP subfolder.
- The output files (1in`YEARS`.tif) will be ready for use in the script input data folder.
- Keep the folder structure: `data/HZD/Country_ISO/type/year/(SSP scenario)/1inYEARS.tif`
- GDAL library must be installed in your Python environment
- Output is exported in the same raster format as the original tiles
- Uses parallel processing for faster execution on multi-core CPUs

## GUI Mode (if available)

In [ ]:
if GUI_AVAILABLE:
    def select_folder():
        root = tk.Tk()
        root.attributes('-topmost', True)
        root.geometry("1x1+0+0")
        root.withdraw()
        root.after(0, lambda: root.focus_force())
        folder_selected = filedialog.askdirectory(master=root)
        root.destroy()
        return folder_selected

    class TilesMosaicingTool:
        def __init__(self):
            self.directory_input = widgets.Text(
                value='',
                placeholder='Enter the directory path',
                description='Directory:',
                disabled=False,
                layout=widgets.Layout(width='500px')
            )
            
            self.search_button = widgets.Button(
                description='Search Folder',
                disabled=False,
                button_style='',
                tooltip='Click to search for a folder',
                layout=widgets.Layout(width='120px')
            )
            
            self.process_button = widgets.Button(
                description='Process Tiles',
                disabled=False,
                button_style='',
                tooltip='Click to start processing',
                layout=widgets.Layout(width='120px')
            )
            
            self.delete_checkbox = widgets.Checkbox(
                value=False,
                description='Delete original subfolders after processing',
                disabled=False,
                indent=False,
                layout=widgets.Layout(width='500px')
            )
            
            self.output = widgets.Output()
            
            self.search_button.on_click(self.on_search_button_clicked)
            self.process_button.on_click(self.on_process_button_clicked)
            
            display(widgets.VBox([
                widgets.HBox([self.directory_input, self.search_button]),
                widgets.HBox([self.process_button, self.delete_checkbox]),
                self.output
            ]))
        
        def on_search_button_clicked(self, b):
            folder_path = select_folder()
            if folder_path:
                self.directory_input.value = folder_path
        
        def on_process_button_clicked(self, b):
            self.output.clear_output()
            with self.output:
                directory = self.directory_input.value
                if not directory or not os.path.isdir(directory):
                    print(f"Error: {directory} is not a valid directory.")
                    return
                
                print(f"Output files will be saved in: {directory}")
                
                subdirs = [os.path.join(directory, subdir) for subdir in os.listdir(directory) 
                           if os.path.isdir(os.path.join(directory, subdir))]
                
                with concurrent.futures.ProcessPoolExecutor() as executor:
                    executor.map(merge_utils.merge_tifs, subdirs)
                
                print("Processing complete. Check the parent directory of each RP subfolder for the merged .tif files.")
                
                if self.delete_checkbox.value:
                    for subdir in subdirs:
                        try:
                            shutil.rmtree(subdir)
                            print(f"Deleted: {subdir}")
                        except Exception as e:
                            print(f"Error deleting {subdir}: {e}")
                    print("Original subfolders have been deleted.")
                else:
                    print("Original subfolders have been kept.")
                
                print("Pre-processing completed.")

    # Create and display the tool
    tool = TilesMosaicingTool()
else:
    print("GUI not available. Use manual mode below.")

## Manual Mode for Tiles Mosaicing

In [ ]:
# Specify the directory containing the downloaded Fathom data (tiles)
# This should be the directory containing RP subfolders (1in5, 1in10, etc.)
mosaic_directory = 'C:/YourWorkDirectory/Fathom3/COUNTRY_ISO/FLOOD_TYPE/2020/'

# Set to True to delete original tile folders after processing
delete_original_folders = False

print(f"Output files will be saved in: {mosaic_directory}")

# Find all subdirectories (return period folders)
subdirs = [os.path.join(mosaic_directory, subdir) 
           for subdir in os.listdir(mosaic_directory) 
           if os.path.isdir(os.path.join(mosaic_directory, subdir))]

print(f"Found {len(subdirs)} subdirectories to process")

# Process subdirectories in parallel
with concurrent.futures.ProcessPoolExecutor() as executor:
    executor.map(merge_utils.merge_tifs, subdirs)

print("Processing complete. Check the parent directory of each RP subfolder for the merged .tif files.")

# Optionally delete original tile folders
if delete_original_folders:
    user_input = input("Are you sure you want to delete the original subfolders? (Y/N): ").strip().lower()
    if user_input == 'y':
        for subdir in subdirs:
            try:
                shutil.rmtree(subdir)
                print(f"Deleted: {subdir}")
            except Exception as e:
                print(f"Error deleting {subdir}: {e}")
        print("Original subfolders have been deleted.")
    else:
        print("Original subfolders have been kept.")
else:
    print("Original subfolders have been kept.")

print("Tiles mosaicing completed.")

---

# Workflow 2: Flood Type Combination

Combine different flood types (fluvial, pluvial, coastal) into a single map using MAX criteria.

## How it works

- Takes multiple flood hazard rasters (e.g., fluvial, pluvial, coastal) for the same return period
- Combines them using a **MAX** operation: for each pixel, takes the maximum valid flood depth
- **Negative values are excluded** from the MAX operation (only valid flood depths ≥0 are compared)
- Creates output covering the union of all input extents
- Handles different spatial extents and CRS automatically

## Helper Functions

In [ ]:
def get_raster_info(raster_path):
    """Extract key metadata from a raster file"""
    with rasterio.open(raster_path) as src:
        return {
            'crs': src.crs,
            'transform': src.transform,
            'nodata': src.nodata,
            'count': src.count,
            'width': src.width,
            'height': src.height,
            'bounds': src.bounds
        }

def calculate_union_bounds(raster_paths):
    """Calculate the union bounds of all input rasters"""
    min_left = float('inf')
    min_bottom = float('inf')
    max_right = float('-inf')
    max_top = float('-inf')
    
    for path in raster_paths:
        with rasterio.open(path) as src:
            left, bottom, right, top = src.bounds
            min_left = min(min_left, left)
            min_bottom = min(min_bottom, bottom)
            max_right = max(max_right, right)
            max_top = max(max_top, top)
    
    return (min_left, min_bottom, max_right, max_top)

def overlay_rasters_max(raster_paths):
    """
    Overlay multiple rasters using MAX criteria.

    CRITICAL: Excludes ALL negative values from MAX computation.
    - Negative values (-32768 nodata, -32767 permanent water) are NOT valid flood depths
    - Only values >= 0 are compared in MAX operation
    - If all inputs are negative at a location, preserves the first non-nodata negative value
    - Creates a new raster that covers the union of all input extents
    """
    # Calculate the union bounds for all input rasters
    union_bounds = calculate_union_bounds(raster_paths)

    # Get reference metadata from the first raster
    with rasterio.open(raster_paths[0]) as src:
        ref_crs = src.crs
        ref_nodata = src.nodata if src.nodata is not None else -32768

        # Calculate dimensions manually based on the resolution and bounds
        x_res, y_res = src.res
        dst_width = int((union_bounds[2] - union_bounds[0]) / x_res)
        dst_height = int((union_bounds[3] - union_bounds[1]) / y_res)

        # Calculate transform for the extent
        dst_transform = rasterio.transform.from_bounds(
            union_bounds[0], union_bounds[1],
            union_bounds[2], union_bounds[3],
            dst_width, dst_height
        )

        # Create metadata for output
        dst_meta = src.meta.copy()
        dst_meta.update({
            'driver': 'GTiff',
            'height': dst_height,
            'width': dst_width,
            'transform': dst_transform,
            'nodata': ref_nodata,
            'compress': 'deflate',
            'zlevel': 9,
            'PREDICTOR': 2
        })

        # Initialize output array with nodata values
        output = np.full((dst_height, dst_width), ref_nodata, dtype=np.float32)

    # Track if we have any valid (non-negative) values at each location
    has_valid = np.zeros((dst_height, dst_width), dtype=bool)

    # Process all rasters
    for idx, path in enumerate(raster_paths):
        with rasterio.open(path) as src:
            # Initialize a destination array for this raster
            src_data = np.full((dst_height, dst_width), ref_nodata, dtype=np.float32)

            # Reproject the current raster to the union bounds
            reproject(
                source=rasterio.band(src, 1),
                destination=src_data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=ref_crs,
                resampling=Resampling.nearest,
                src_nodata=src.nodata,
                dst_nodata=ref_nodata
            )

            # CRITICAL: Only consider non-negative values for MAX operation
            # Negative values (nodata, permanent water) are EXCLUDED from computation
            valid_mask = src_data >= 0

            if idx == 0:
                # Initialize output with first raster
                output = src_data.copy()
                has_valid = valid_mask.copy()
            else:
                # Where src has valid (non-negative) data
                where_src_valid = valid_mask
                # Where output already has valid (non-negative) data
                where_output_valid = has_valid

                # Where both have valid data: take MAX
                both_valid = where_src_valid & where_output_valid
                output[both_valid] = np.maximum(output[both_valid], src_data[both_valid])

                # Where only src has valid data: use src value
                only_src_valid = where_src_valid & (~where_output_valid)
                output[only_src_valid] = src_data[only_src_valid]
                has_valid[only_src_valid] = True

                # Where only output has valid data: keep output (no change)
                # Where neither has valid data: check for negative values to preserve
                neither_valid = (~where_src_valid) & (~where_output_valid)
                # If output is still nodata but src has a negative value (like permanent water)
                is_nodata_output = output == ref_nodata
                has_negative_src = (src_data < 0) & (src_data != ref_nodata)
                preserve_negative = neither_valid & is_nodata_output & has_negative_src
                output[preserve_negative] = src_data[preserve_negative]

    return output, dst_meta

print("Flood type combination functions loaded")

## Combine Flood Types Example

Replace the paths below with your actual raster file paths.

In [ ]:
# Replace these paths with your actual raster file paths
# Example: combining fluvial, pluvial, and coastal flood types for 1in100 return period
raster_paths = [
    'data/HZD/COUNTRY_ISO/FLUVIAL_UNDEFENDED/2050/SSP5_8.5/1in100.tif',
    'data/HZD/COUNTRY_ISO/PLUVIAL_DEFENDED/2050/SSP5_8.5/1in100.tif',
    'data/HZD/COUNTRY_ISO/COASTAL_UNDEFENDED/2050/SSP5_8.5/1in100.tif',
]

# Display input raster information
print("Input Raster Information:")
for path in raster_paths:
    if os.path.exists(path):
        print(f"\nRaster: {Path(path).name}")
        info = get_raster_info(path)
        for key, value in info.items():
            print(f"{key}: {value}")
    else:
        print(f"\nWarning: {path} does not exist")

# Perform overlay operation
print("\nPerforming MAX overlay operation (excluding negative values)...")
result, metadata = overlay_rasters_max(raster_paths)
print("Overlay complete!")

# Save the result
output_path = 'data/HZD/COUNTRY_ISO/COMBINED/2050/SSP5_8.5/1in100.tif'
print(f"\nSaving result to: {output_path}")

# Create output directory if needed
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Write output raster
with rasterio.open(output_path, 'w', **metadata) as dst:
    dst.write(result.astype(rasterio.float32), 1)

print("Save complete!")

# Display output information
print("\nOutput Raster Information:")
output_info = get_raster_info(output_path)
for key, value in output_info.items():
    print(f"{key}: {value}")

# Visualize results (optional)
plt.figure(figsize=(12, 8))
# Mask negative values for visualization
display_data = np.where(result >= 0, result, np.nan)
plt.imshow(display_data, cmap='viridis')
plt.colorbar(label='Flood Depth (cm)')
plt.title('Combined Flood Hazard Map (MAX of all types)')
plt.show()

---

# Workflow 3: Resampling

Resample TIFF files from 30m (≈0.000278°) to 90m (≈0.000833°) resolution.

## Features

- Parallel processing for faster execution
- Preserves the original data format and CRS
- Uses nearest-neighbor resampling to preserve exact values
- Names output files with "_90m" suffix
- **Properly handles negative nodata values** (excluded from resampling interpolation)

## Set Parameters

In [ ]:
# Set folder path to search for TIFF files
# Replace this with your actual folder path
resample_folder_path = 'data/HZD/COUNTRY_ISO/FLOOD_TYPE/2020'

# Define target resolution (90m ≈ 0.000833333 degrees)
target_resolution = 0.000833333

# Output suffix
output_suffix = "_90m"

# Create output folder if it doesn't exist
output_folder = os.path.join(resample_folder_path, 'resampled')
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created output folder: {output_folder}")
else:
    print(f"Output folder already exists: {output_folder}")

# Set number of parallel processes (None = use all available cores)
num_workers = multiprocessing.cpu_count() - 1  # Leave one core free
print(f"Using {num_workers} worker processes for parallel execution")

## Find TIFF Files

In [ ]:
# Find all TIFF files in the folder
tiff_files = tiff_resampler.find_tiff_files(resample_folder_path)

print(f"Found {len(tiff_files)} TIFF files:")
for file in tiff_files:
    print(f"  - {os.path.basename(file)}")

## Check Resolutions of Found Files

In [ ]:
# Check resolution of each file
resolution_data = []
for file in tiff_files:
    try:
        resolution = tiff_resampler.check_resolution(file)
        resolution_data.append({
            'file': os.path.basename(file),
            'x_resolution': resolution[0],
            'y_resolution': resolution[1]
        })
        print(f"{os.path.basename(file)}: {resolution}")
    except Exception as e:
        print(f"Error reading {os.path.basename(file)}: {e}")

# Display as a table if there are files
if resolution_data:
    display(pd.DataFrame(resolution_data))

## Resample Files in Parallel

In [ ]:
# Measure performance
start_time = time.time()

# Process files in parallel
print(f"Starting parallel processing of {len(tiff_files)} files with {num_workers} workers...")
results = tiff_resampler.resample_tiffs_parallel(
    folder_path=resample_folder_path,
    target_resolution=target_resolution,
    output_suffix=output_suffix,
    max_workers=num_workers
)

# Calculate elapsed time
elapsed_time = time.time() - start_time
print(f"Processing completed in {elapsed_time:.2f} seconds")

## Results Summary

In [ ]:
# Count successes and failures
successes = sum(1 for r in results if r['status'] == 'Success')
failures = len(results) - successes

# Display results summary
print("\nProcessing Results:")
print("-" * 80)
for result in results:
    if result['status'] == 'Success':
        print(f"✓ {result['input_file']} -> {result['output_file']}")
        print(f"  Original resolution: {result['original_resolution']}")
        print(f"  New resolution: {result['new_resolution']}")
    else:
        print(f"✗ {result['input_file']}: {result['status']}")
    print("-" * 80)

print(f"\nSummary: {successes} files processed successfully, {failures} failed")

# Create a DataFrame for better visualization
if results:
    df_results = pd.DataFrame([
        {
            'File': r['input_file'],
            'Output': r.get('output_file', 'N/A'),
            'Status': 'Success' if r['status'] == 'Success' else r['status'],
            'Original Resolution': r.get('original_resolution', 'N/A') if r['status'] == 'Success' else 'N/A',
            'New Resolution': r.get('new_resolution', 'N/A') if r['status'] == 'Success' else 'N/A'
        } for r in results
    ])
    
    display(df_results)

## Verify Output Files

In [ ]:
# Verify output files have the correct resolution
verification_results = tiff_resampler.verify_output_files(output_folder, target_resolution)

print(f"\nVerifying {len(verification_results)} output files:")
for result in verification_results:
    if result['status'] == 'Verified':
        match_status = "✓" if result['matches_target'] else "✗"
        print(f"{match_status} {result['file']}: Resolution = {result['resolution']}")
    else:
        print(f"✗ {result['file']}: {result['status']}")

# Create a DataFrame for better visualization
if verification_results:
    df_verify = pd.DataFrame([
        {
            'File': r['file'],
            'Resolution': r.get('resolution', 'N/A') if r['status'] == 'Verified' else 'N/A',
            'Matches Target': r.get('matches_target', False) if r['status'] == 'Verified' else False,
            'Status': r['status']
        } for r in verification_results
    ])
    
    display(df_verify)

## Performance Analysis

In [ ]:
# Calculate some performance metrics
if len(tiff_files) > 0 and elapsed_time > 0:
    avg_time_per_file = elapsed_time / len(tiff_files)
    print(f"Average processing time per file: {avg_time_per_file:.2f} seconds")
    print(f"Files processed per second: {len(tiff_files) / elapsed_time:.2f}")
    print(f"Estimated time savings with parallel processing: {len(tiff_files) * avg_time_per_file - elapsed_time:.2f} seconds")

---

## Summary

This notebook provides three essential Fathom 3 pre-processing workflows:

1. **Tiles Mosaicing**: Merges downloaded tiles into unified rasters
2. **Flood Type Combination**: Combines multiple flood types using MAX (excluding negative values)
3. **Resampling**: Changes resolution from 30m to 90m while preserving data integrity

All workflows properly handle Fathom 3's data encoding:
- Values 0-1,000: Valid flood depths in centimetres
- Value −32,767: Permanent water bodies
- Value −32,768: NoData sentinel
- Negative values excluded from all computations